# t-SNE Diagnosis: x86_64 → MIPS (roberta_20 + GAT)

審查人要求的診斷分析：
1. **Before DA** — GAT (trained on source only) embedding，看 x86_64 vs MIPS 的分佈偏移
2. **After DA** — CCSA embedding，看 domain adaptation 後的對齊程度
3. 每個 class 各取固定樣本數，保持圖面可讀

In [7]:
import sys
sys.path.append("/home/tommy/Project/PCBSDA/ours")

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.manifold import TSNE
from torch_geometric.loader import DataLoader

from configs.ccsa.run_all_config import get_run_all_config
from src.transfer_learning.ccsa.models import GAT_CCSA
from src.transfer_learning.ccsa.utils import prepare_ccsa_data

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


In [8]:
BASE_PATH  = "/home/tommy/Project/PCBSDA"
EMBEDDING  = "roberta_20"
MODEL_DIR  = f"{BASE_PATH}/ours/outputs/models/ccsa/{EMBEDDING}/gat/x86_64_to_MIPS"
SAVE_DIR   = Path(f"{BASE_PATH}/ours/outputs/plots/ccsa/{EMBEDDING}/gat/tsne_mips")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 每個 (domain, family) 最多取幾個點，保持圖面可讀
MAX_PER_GROUP = 30

FAMILIES = ['dnsamp', 'dofloo', 'gafgyt', 'kaiji', 'meterpreter', 'mirai', 'mobidash', 'tsunami']

DOMAIN_COLOR = {
    'source': '#2166AC',   # blue  — x86_64
    'target': '#D6604D',   # red   — MIPS
}
FAMILY_MARKER = {
    'dnsamp':      'o',
    'dofloo':      's',
    'gafgyt':      '^',
    'kaiji':       'D',
    'meterpreter': 'P',
    'mirai':       'X',
    'mobidash':    'v',
    'tsunami':     'h',
}

print("Config loaded.")
print(f"Saving plots to: {SAVE_DIR}")

Config loaded.
Saving plots to: /home/tommy/Project/PCBSDA/ours/outputs/plots/ccsa/roberta_20/gat/tsne_mips


## Load Data (x86_64 → MIPS, roberta_20)

In [9]:
base_cfg = get_run_all_config()

# Patch config for x86_64 -> MIPS
cfg = dict(base_cfg)
cfg['source_cpus'] = ['x86_64']
cfg['target_cpus'] = ['MIPS']

src_tag = 'x86_64'
tgt_tag = 'MIPS'
cfg['source_cache_file'] = f"{base_cfg['cache_dir']}/source_{src_tag}.pkl"
cfg['target_cache_file'] = f"{base_cfg['cache_dir']}/target_{tgt_tag}.pkl"

# Use seed=42 for data loading (deterministic split)
RANDOM_STATE = 42

src_train, src_val, tgt_train, tgt_test, label_encoder, num_classes = prepare_ccsa_data(cfg, RANDOM_STATE)

source_all = src_train + src_val
target_all = tgt_train + tgt_test
fewshot_idx = set(range(len(tgt_train)))

print(f"Source (x86_64): {len(source_all)}")
print(f"Target (MIPS)  : {len(target_all)}  (few-shot: {len(tgt_train)}, test: {len(tgt_test)})")
print(f"Classes: {list(label_encoder.classes_)}")

載入快取: /home/tommy/Project/PCBSDA/ours/outputs/cache/ccsa/roberta_20/source_x86_64.pkl
載入快取: /home/tommy/Project/PCBSDA/ours/outputs/cache/ccsa/roberta_20/target_MIPS.pkl

CCSA Data Summary:
  Source train: 1664, Source val: 417
  Target train (few-shot): 40, Target test: 1735
  Num classes: 8
  Source train distribution: {4: 240, 5: 240, 2: 240, 0: 166, 7: 240, 1: 103, 6: 240, 3: 195}
  Target train distribution: {0: 5, 1: 5, 2: 5, 3: 5, 4: 5, 5: 5, 6: 5, 7: 5}
  Target test distribution: {5: 295, 2: 295, 7: 295, 6: 295, 1: 295, 3: 159, 4: 42, 0: 59}
Source (x86_64): 2081
Target (MIPS)  : 1775  (few-shot: 40, test: 1735)
Classes: [np.str_('dnsamp'), np.str_('dofloo'), np.str_('gafgyt'), np.str_('kaiji'), np.str_('meterpreter'), np.str_('mirai'), np.str_('mobidash'), np.str_('tsunami')]


## Helpers

In [ ]:
def build_meta(source_graphs, target_graphs, fewshot_indices, label_encoder):
    meta = []
    for g in source_graphs:
        meta.append({'domain': 'source', 'family': label_encoder.classes_[int(g.y)], 'fewshot': False})
    for i, g in enumerate(target_graphs):
        meta.append({'domain': 'target', 'family': label_encoder.classes_[int(g.y)], 'fewshot': i in fewshot_indices})
    return meta


def subsample_graphs(graphs, meta, max_per_class=50, seed=0):
    """每個 (domain, family) 最多保留 max_per_class 個樣本，減少記憶體用量。"""
    rng = np.random.RandomState(seed)
    from collections import defaultdict
    buckets = defaultdict(list)
    for i, m in enumerate(meta):
        buckets[(m['domain'], m['family'])].append(i)

    keep = []
    for idx_list in buckets.values():
        if len(idx_list) > max_per_class:
            idx_list = rng.choice(idx_list, size=max_per_class, replace=False).tolist()
        keep.extend(idx_list)
    keep = sorted(keep)
    return [graphs[i] for i in keep], [meta[i] for i in keep]


@torch.no_grad()
def extract_ccsa_embeddings(model, graphs, batch_size=64, use_cpu=True):
    """GAT_CCSA.forward returns (pred, feature); we want feature.
    use_cpu=True 把推論放在 CPU 以避免 OOM。"""
    infer_device = torch.device('cpu') if use_cpu else DEVICE
    model_dev = model.to(infer_device)
    model_dev.eval()
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    feats = []
    for batch in loader:
        batch = batch.to(infer_device)
        _, feature = model_dev(batch.x, batch.edge_index, batch.batch)
        feats.append(feature.cpu().numpy())
    model.to(DEVICE)   # 推完移回 GPU
    return np.concatenate(feats)


@torch.no_grad()
def extract_raw_embeddings(graphs, batch_size=128):
    """Global mean pool of raw node features — CPU only, no model."""
    from torch_geometric.nn import global_mean_pool
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    feats = []
    for batch in loader:
        pooled = global_mean_pool(batch.x, batch.batch)
        feats.append(pooled.numpy())
    return np.concatenate(feats)


def run_tsne(embeddings, perplexity=30, seed=0):
    print(f"Running t-SNE on {embeddings.shape} ...")
    return TSNE(n_components=2, perplexity=perplexity, random_state=seed,
                max_iter=1000).fit_transform(embeddings)


print("Helpers defined.")

In [11]:
def plot_tsne(coords, meta, title, save_path, show_fewshot=True,
              max_per_group=MAX_PER_GROUP):
    rng = np.random.RandomState(0)
    domains_arr  = np.array([m['domain']  for m in meta])
    families_arr = np.array([m['family']  for m in meta])
    fewshot_arr  = np.array([m['fewshot'] for m in meta])

    fig, ax = plt.subplots(figsize=(10, 7))

    for domain in ('source', 'target'):
        for fam in FAMILIES:
            mask = (domains_arr == domain) & (families_arr == fam)
            if show_fewshot and domain == 'target':
                mask = mask & ~fewshot_arr
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue
            if len(idx) > max_per_group:
                idx = rng.choice(idx, size=max_per_group, replace=False)
            ax.scatter(coords[idx, 0], coords[idx, 1],
                       c=DOMAIN_COLOR[domain],
                       marker=FAMILY_MARKER[fam],
                       s=50, alpha=0.7,
                       edgecolors='white', linewidths=0.4)

    # Few-shot samples — gold stars
    if show_fewshot:
        for fam in FAMILIES:
            idx = np.where(fewshot_arr & (families_arr == fam))[0]
            if len(idx) == 0:
                continue
            ax.scatter(coords[idx, 0], coords[idx, 1],
                       c='#FFD700', marker=FAMILY_MARKER[fam],
                       s=120, alpha=1.0,
                       edgecolors='#333', linewidths=0.7, zorder=5)

    # Legend — domain
    domain_handles = [
        mpatches.Patch(color=DOMAIN_COLOR['source'], label='Source (x86_64)'),
        mpatches.Patch(color=DOMAIN_COLOR['target'], label='Target (MIPS)'),
    ]
    if show_fewshot:
        domain_handles.append(
            mpatches.Patch(color='#FFD700', label='Target few-shot (5/class)')
        )
    # Legend — family (markers)
    family_handles = [
        plt.Line2D([0], [0], marker=FAMILY_MARKER[f], color='#555',
                   linestyle='None', markersize=8, markerfacecolor='#555', label=f)
        for f in FAMILIES
    ]

    leg1 = ax.legend(handles=domain_handles, title='Domain',
                     loc='upper left', fontsize=9, title_fontsize=9,
                     framealpha=0.9, edgecolor='#ccc')
    ax.add_artist(leg1)
    ax.legend(handles=family_handles, title='Family',
              loc='upper right', fontsize=9, title_fontsize=9,
              framealpha=0.9, edgecolor='#ccc')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE dim 1', fontsize=10)
    ax.set_ylabel('t-SNE dim 2', fontsize=10)
    ax.grid(True, alpha=0.12, linestyle='--')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved: {save_path}")


print("Plot function defined.")

Plot function defined.


## Plot 1 — Before DA (raw node feature mean pool)

In [ ]:
all_graphs = source_all + target_all
meta_all   = build_meta(source_all, target_all, fewshot_idx, label_encoder)

# 每個 (domain, family) 最多 50 個，few-shot 全保留
graphs_sub, meta_sub = subsample_graphs(all_graphs, meta_all, max_per_class=50)
print(f"Subsampled: {len(graphs_sub)} graphs (from {len(all_graphs)})")

raw_emb   = extract_raw_embeddings(graphs_sub)
raw_coord = run_tsne(raw_emb)

plot_tsne(
    raw_coord, meta_sub,
    title="t-SNE: Raw Node Feature (no GNN, no DA)  |  x86_64 → MIPS",
    save_path=SAVE_DIR / "tsne_raw_x86_64_to_MIPS.png",
    show_fewshot=False,
)

## Plot 2 — After CCSA (seed=42)

In [ ]:
model_path = f"{MODEL_DIR}/ccsa_best_rs42.pt"

model = GAT_CCSA(
    num_node_features=cfg['num_node_features'],
    hidden_channels=cfg['hidden_channels'],
    output_channels=cfg['output_channels'],
    num_classes=num_classes,
    num_layers=cfg['num_layers'],
    dropout=cfg['dropout'],
    pooling=cfg['pooling'],
    heads=cfg.get('gat_heads', 4),
).to(DEVICE)

state = torch.load(model_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(state)
print(f"Loaded: {model_path}")

# graphs_sub / meta_sub 已在 Plot 1 建好，直接重用
ccsa_emb   = extract_ccsa_embeddings(model, graphs_sub, use_cpu=True)
ccsa_coord = run_tsne(ccsa_emb)

plot_tsne(
    ccsa_coord, meta_sub,
    title="t-SNE: CCSA Embedding (after DA, seed=42)  |  x86_64 → MIPS",
    save_path=SAVE_DIR / "tsne_ccsa_rs42_x86_64_to_MIPS.png",
    show_fewshot=True,
)

## Plot 3 — After CCSA (seed=123, worst seed)

In [ ]:
# Reload data for seed=123 (different few-shot split)
src_train_123, src_val_123, tgt_train_123, tgt_test_123, _, _ = prepare_ccsa_data(cfg, 123)
source_123 = src_train_123 + src_val_123
target_123 = tgt_train_123 + tgt_test_123
fewshot_123 = set(range(len(tgt_train_123)))
meta_123    = build_meta(source_123, target_123, fewshot_123, label_encoder)
all_graphs_123 = source_123 + target_123

graphs_sub_123, meta_sub_123 = subsample_graphs(all_graphs_123, meta_123, max_per_class=50)
print(f"Subsampled: {len(graphs_sub_123)} graphs")

model_path_123 = f"{MODEL_DIR}/ccsa_best_rs123.pt"
state_123 = torch.load(model_path_123, map_location=DEVICE, weights_only=False)
model.load_state_dict(state_123)
print(f"Loaded: {model_path_123}")

ccsa_emb_123   = extract_ccsa_embeddings(model, graphs_sub_123, use_cpu=True)
ccsa_coord_123 = run_tsne(ccsa_emb_123)

plot_tsne(
    ccsa_coord_123, meta_sub_123,
    title="t-SNE: CCSA Embedding (after DA, seed=123)  |  x86_64 → MIPS",
    save_path=SAVE_DIR / "tsne_ccsa_rs123_x86_64_to_MIPS.png",
    show_fewshot=True,
)

## Per-class F1 Summary across all MIPS pairs

從 logs 整理出來的數字，看哪個 class 在 MIPS 上持續表現最差。

In [ ]:
import pandas as pd

# Per-class F1 from logs (x86_64 -> MIPS, 3 seeds)
data_x86_mips = {
    'family':      ['dnsamp', 'dofloo', 'gafgyt', 'kaiji', 'meterpreter', 'mirai', 'mobidash', 'tsunami'],
    'support':     [59,        295,      295,       159,     42,            295,      295,         295],
    'f1_rs42':     [0.94,      0.90,     0.78,      1.00,    0.98,          0.78,     0.87,        0.82],
    'f1_rs123':    [0.67,      0.99,     0.70,      1.00,    0.99,          0.74,     1.00,        0.66],
    'f1_rs7':      [0.85,      0.93,     0.85,      1.00,    1.00,          0.81,     1.00,        0.87],
}
df_x86 = pd.DataFrame(data_x86_mips)
df_x86['f1_mean'] = df_x86[['f1_rs42','f1_rs123','f1_rs7']].mean(axis=1).round(3)
df_x86['f1_std']  = df_x86[['f1_rs42','f1_rs123','f1_rs7']].std(axis=1).round(3)
df_x86 = df_x86.sort_values('f1_mean')

print("=== x86_64 → MIPS per-class F1 ===")
print(df_x86[['family','support','f1_mean','f1_std','f1_rs42','f1_rs123','f1_rs7']].to_string(index=False))

In [ ]:
# ARM-32 -> MIPS per-class F1
data_arm_mips = {
    'family':      ['dnsamp', 'dofloo', 'gafgyt', 'kaiji', 'meterpreter', 'mirai', 'mobidash', 'tsunami'],
    'support':     [59,        295,      295,       159,     42,            295,      295,         295],
    'f1_rs42':     [0.98,      0.99,     0.86,      1.00,    0.99,          0.86,     1.00,        0.93],
    'f1_rs123':    [0.96,      0.99,     0.75,      1.00,    0.99,          0.71,     0.98,        0.66],
    'f1_rs7':      [0.98,      0.99,     0.81,      1.00,    0.99,          0.85,     0.99,        0.85],
}
df_arm = pd.DataFrame(data_arm_mips)
df_arm['f1_mean'] = df_arm[['f1_rs42','f1_rs123','f1_rs7']].mean(axis=1).round(3)
df_arm['f1_std']  = df_arm[['f1_rs42','f1_rs123','f1_rs7']].std(axis=1).round(3)
df_arm = df_arm.sort_values('f1_mean')

# Intel -> MIPS per-class F1
data_intel_mips = {
    'family':      ['dnsamp', 'dofloo', 'gafgyt', 'kaiji', 'meterpreter', 'mirai', 'mobidash', 'tsunami'],
    'support':     [59,        295,      295,       159,     42,            295,      295,         295],
    'f1_rs42':     [0.90,      0.99,     0.84,      1.00,    0.85,          0.84,     0.98,        0.88],
    'f1_rs123':    [0.94,      0.99,     0.71,      1.00,    1.00,          0.69,     0.99,        0.73],
    'f1_rs7':      [0.88,      1.00,     0.81,      1.00,    0.66,          0.86,     0.99,        0.85],
}
df_intel = pd.DataFrame(data_intel_mips)
df_intel['f1_mean'] = df_intel[['f1_rs42','f1_rs123','f1_rs7']].mean(axis=1).round(3)
df_intel['f1_std']  = df_intel[['f1_rs42','f1_rs123','f1_rs7']].std(axis=1).round(3)
df_intel = df_intel.sort_values('f1_mean')

print("=== ARM-32 → MIPS per-class F1 ===")
print(df_arm[['family','support','f1_mean','f1_std']].to_string(index=False))
print()
print("=== Intel → MIPS per-class F1 ===")
print(df_intel[['family','support','f1_mean','f1_std']].to_string(index=False))

In [ ]:
# 三個遷移對的 per-class 平均 F1 比較圖
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

pairs = [
    (df_x86,   'x86_64 → MIPS'),
    (df_arm,   'ARM-32 → MIPS'),
    (df_intel, 'Intel → MIPS'),
]

COLORS = ['#D6604D' if m < 0.80 else '#4393C3' for m in df_x86['f1_mean']]

for ax, (df, title) in zip(axes, pairs):
    colors = ['#D6604D' if m < 0.80 else '#4393C3' for m in df['f1_mean']]
    bars = ax.barh(df['family'], df['f1_mean'], color=colors, xerr=df['f1_std'],
                   capsize=4, error_kw={'elinewidth':1.2})
    ax.axvline(0.80, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlim(0.5, 1.05)
    ax.set_xlabel('F1-macro', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    for bar, val in zip(bars, df['f1_mean']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=8)

plt.suptitle('Per-class F1 for all → MIPS pairs  (red = F1 < 0.80)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_path = SAVE_DIR / "perclass_f1_mips_all_pairs.png"
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {save_path}")

## Summary

從 logs 整理的觀察：
- **gafgyt** 和 **mirai** 在所有三個 → MIPS 遷移對中持續 F1 < 0.85，是最難遷移的家族
- **tsunami** 在 x86_64 → MIPS 的 seed=123 中 F1 僅 0.66，variance 很大
- **kaiji / mobidash / dofloo** 在所有對上幾乎完美（F1 > 0.95）
- **meterpreter** support 僅 42，Intel → MIPS seed=7 時 F1 只有 0.66（precision=0.49），可能是 support 太小導致不穩定